# 03 · Busca híbrida: BM25 + vetorial com RRF

> **Trilha do projeto:** 01 Base → 02 Índice vetorial → **03 Busca híbrida** → 04 Guardrail → 05 Roteador → 06 Multi-Query → 07 Evidência e recusa → 08 Geração → 09 Grafo RAG → 10 Avaliação

**Contexto.** O notebook 02 mostrou que a busca vetorial entende paráfrases, mas tem
dois limites: sempre devolve um top-k (mesmo sem evidência) e pode subestimar termos
exatos, como "nota fiscal" ou "boleto". Aqui juntamos a ela o **BM25**, a busca por
palavras, e fundimos os dois rankings com **RRF** (*Reciprocal Rank Fusion*). É o último
notebook da **base de conhecimento**: ao final, o `Buscador` está pronto para o RAG.

**Objetivo deste notebook**

- entender o BM25 e por que a pergunta passa pela mesma normalização da base;
- comparar vetorial e BM25 lado a lado e ver onde cada um tropeça;
- fundir os dois com RRF e aplicar filtros de negócio às duas buscas;
- medir a precisão@10 das três buscas com dez perguntas de analista;
- encontrar um primeiro sinal para **recusar** perguntas sem evidência.

| Seção | Pergunta que responde | Função em `src/voc_rag` |
|---|---|---|
| 0 | Preparação: base, índices e buscador | `busca.Buscador` |
| 1 | Como funciona a busca por palavras? | `busca.tokenizar_lexical`, `IndiceBM25` |
| 2 | Onde cada busca acerta e erra? | `Buscador.buscar(modo=..., min_palavras=...)` |
| 3 | Como juntar dois rankings? | `busca.rrf` |
| 4 | Como filtrar por nota, categoria, período? | `busca.posicoes_permitidas` |
| 5 | A recuperação está boa? | `avaliacao.checar_recuperacao` |
| 6 | Dá para saber quando não há evidência? | `avaliacao.checar_recusa` |

> **Para executar:** rode antes os notebooks 01 e 02 (ou `python -m voc_rag.preparacao`
> e `python -m voc_rag.indice`). Não usa LLM.

In [1]:
import inspect
import time

import pandas as pd

from voc_rag import avaliacao, busca, config, indice
from voc_rag.preparacao import carregar_base

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)


def mostrar_codigo(funcao):
    """Imprime o código-fonte de uma função do src/, para lermos junto com o resultado."""
    print(inspect.getsource(funcao))


documentos = carregar_base()
indice_vetorial, manifesto = indice.carregar_indice()
modelo = indice.carregar_modelo()

inicio = time.perf_counter()
buscador = busca.Buscador(documentos, indice_vetorial, modelo)
print(f"{len(documentos):,} documentos | BM25 montado em {time.perf_counter() - inicio:.1f} s "
      f"| índice vetorial de {manifesto['criado_em']}")

COLUNAS = ["pontuacao", "similaridade", "pos_vetorial", "pos_bm25", "nota", "categoria", "texto"]

W1106 21:02:55.168000 79076 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

42,138 documentos | BM25 montado em 0.3 s | índice vetorial de 2026-10-28T12:05:30+00:00


## Seção 1 · BM25: a busca por palavras

O BM25 é um TF-IDF com dois ajustes:

- **TF** (frequência do termo na review) conta, mas **satura** (parâmetro `k1`):
  repetir "péssimo" cinco vezes não vale cinco vezes mais.
- **IDF** premia termos **raros** na base: "fiscal" pesa muito mais que "produto".
- O tamanho da review é **normalizado** (parâmetro `b`): uma review longa não ganha só
  por ter mais palavras.

Ele roda sobre o `texto_lexical` do notebook 01 (minúsculo, sem acento, contrações
expandidas). **A pergunta passa pela mesma normalização**; senão "Não" na pergunta não
casaria com "nao" na review. Observe os termos da consulta abaixo.

In [2]:
mostrar_codigo(busca.tokenizar_lexical)
mostrar_codigo(busca.IndiceBM25.buscar)

from voc_rag.texto import limpar_normalizar
pergunta = "Vc NÃO mandou a Nota Fiscal!!"
print("termos que o BM25 procura:", busca.tokenizar_lexical(limpar_normalizar(pergunta, remove_acentos=True)))

def tokenizar_lexical(texto):
    """Tokenização simples para o BM25: minúsculas + palavras alfanuméricas."""
    return re.findall(r"[a-z0-9À-ÿ\-]+", texto.lower())

    def buscar(self, consulta, k=10, permitidos=None):
        """Top-k como lista de (posição, pontuação BM25). Documentos sem nenhum termo ficam de fora.

        A consulta passa pela MESMA normalização da base (sem acento, contrações
        expandidas), senão "não" na pergunta não casaria com "nao" na review.
        """
        termos = tokenizar_lexical(limpar_normalizar(consulta, remove_acentos=True))
        pontuacoes = self.bm25.get_scores(termos)
        if permitidos is not None:
            mascara = np.zeros(len(pontuacoes), dtype=bool)
            mascara[np.asarray(permitidos, dtype="int64")] = True
            pontuacoes = np.where(mascara, pontuacoes, 0.0)
        ordem = np.argsort(-pontuacoes)[:k]
        return [(int(p), float(pontuacoes[p])) for p in ordem if pontuacoes[p] > 0]

termos que o BM25 pr

In [3]:
buscador.buscar("nota fiscal", k=5, modo="bm25")[["bm25", "nota", "categoria", "texto"]]

,bm25,nota,categoria,texto
rank,,,,
1,12.14,4,informatica_acessorios,Faltou a nota fiscal
2,12.14,2,cool_stuff,Preciso da nota fiscal
3,11.51,3,moveis_decoracao,A nota fiscal não veio.
4,11.51,4,cama_mesa_banho,Não veio a nota fiscal
5,11.51,3,moveis_escritorio,Ficou faltando a nota fiscal


**Conclusão da Seção 1:** "Vc NÃO mandou a Nota Fiscal!!" vira `voce nao mandou a
nota fiscal`, e as cinco primeiras reviews do BM25 contêm literalmente "nota fiscal",
que a busca vetorial tinha misturado com "cupom fiscal".

## Seção 2 · Vetorial x BM25 lado a lado

Duas perguntas que puxam para lados opostos:

1. **Paráfrase:** "o pedido nunca foi entregue". Os clientes escrevem "não chegou",
   "ainda não recebi". A vetorial acha por sentido; o BM25 depende de as palavras da
   pergunta aparecerem na review.
2. **Termo exato:** "boleto". Palavra específica, rara e decisiva.

**Observe** as colunas `similaridade` e `bm25`: cada busca é cega para o critério da
outra.

In [4]:
for pergunta in ["o pedido nunca foi entregue", "boleto"]:
    for modo in ["vetorial", "bm25"]:
        resultado = buscador.buscar(pergunta, k=5, modo=modo)
        print(f'\n"{pergunta}" — {modo.upper()} ({resultado.attrs["latencia_ms"]:.0f} ms)')
        print(resultado[["similaridade", "bm25", "nota", "texto"]].to_string())


"o pedido nunca foi entregue" — VETORIAL (20 ms)
      similaridade  bm25  nota                                                                        texto
rank                                                                                                       
1            0.845   0.0     1                                                    o pedido nao foi entregue
2            0.819   0.0     1                                                    O pedido não foi entregue
3            0.759   0.0     1                                            Meus pedidos não foram entregues.
4            0.746   0.0     5                                                Não foi entregue o meu pedido
5            0.742   0.0     1  Apesar de constar como entregue no histórico de pedidos, nada foi entregue.

"o pedido nunca foi entregue" — BM25 (21 ms)
      similaridade   bm25  nota                                                                                 texto
rank                          

### Um tropeço da busca vetorial: reviews curtas demais

A pergunta "boleto" traz na busca vetorial reviews como "Certinho" e "ecelente", com
similaridade **0,95**, sem nenhuma relação. Com uma pergunta de **uma palavra**, o
embedding dela fica perto dos embeddings de reviews de **uma palavra**, qualquer que
seja o assunto. Cerca de 14% da base tem 1 ou 2 palavras.

**Decisão:** o `Buscador` só usa como **evidência** reviews com pelo menos
`config.MIN_PALAVRAS_EVIDENCIA` (3) palavras. Elas **continuam na base** para contagens
e temas; só não são citadas. O parâmetro `min_palavras=1` desliga o filtro; compare
abaixo.

**O que o filtro não resolve.** Com o filtro ligado, o lado vetorial de "boleto" ainda
traz elogios genéricos ("Fooi bem entregue", ~0,84), e na híbrida eles ficam
intercalados com os acertos do BM25. Uma pergunta de **uma palavra** é fraca para o
embedding. Isso se resolve no pipeline RAG: a **reformulação da pergunta** (Multi-Query,
notebook 06) transforma "boleto" em frases completas, e a **checagem de evidência**
(notebook 07) descarta o que não fala do assunto.

In [5]:
for min_palavras in [1, config.MIN_PALAVRAS_EVIDENCIA]:
    resultado = buscador.buscar("boleto", k=5, modo="vetorial", min_palavras=min_palavras)
    print(f'\n"boleto" — VETORIAL com min_palavras={min_palavras}')
    print(resultado[["similaridade", "nota", "texto"]].to_string())
print(f"\nReviews com menos de {config.MIN_PALAVRAS_EVIDENCIA} palavras: "
      f"{(documentos['n_palavras'] < config.MIN_PALAVRAS_EVIDENCIA).sum():,}")


"boleto" — VETORIAL com min_palavras=1
      similaridade  nota      texto
rank                               
1            0.959     5   Certinho
2            0.954     5   ecelente
3            0.950     5     botimo
4            0.942     5  Esxelente
5            0.941     4  escelente



"boleto" — VETORIAL com min_palavras=3
      similaridade  nota                                        texto
rank                                                                 
1            0.842     5                            Fooi bem entregue
2            0.826     5                             MUITO OBRIGADA !
3            0.822     5              CONTENTE. CONTENTE COM AENTREGA
4            0.821     5  RECOMENDO. OTIMO ENTREGOU NO PRAZO PARABENS
5            0.819     1     Otima. Muito Agel com a entrega obrigado

Reviews com menos de 3 palavras: 5,983


## Seção 3 · Fusão com RRF

As pontuações das duas buscas estão em **escalas diferentes** (cosseno entre 0 e 1;
BM25 sem teto), então não dá para somá-las. O **Reciprocal Rank Fusion** usa só a
**posição** em cada ranking:

$$\text{RRF}(d) = \sum_{\text{rankings}} \frac{1}{k + \text{posição}(d)}, \quad k = 60$$

Quem aparece bem colocado **nas duas** buscas sobe; quem aparece em só uma ainda entra,
mais abaixo. O `k = 60` amortece a diferença entre 1º e 2º lugar.

In [6]:
mostrar_codigo(busca.rrf)

# Exemplo de brinquedo: o documento 7 é 2º num ranking e 1º no outro
print(busca.rrf([[3, 7, 9], [7, 1, 3]]))

def rrf(rankings, k=RRF_K):
    """Reciprocal Rank Fusion: funde rankings somando 1/(k + posição)."""
    pontos = {}
    for ranking in rankings:
        for posicao, idx in enumerate(ranking):
            pontos[idx] = pontos.get(idx, 0.0) + 1.0 / (k + posicao + 1)
    return sorted(pontos.items(), key=lambda par: -par[1])

[(7, 0.03252247488101534), (3, 0.032266458495966696), (1, 0.016129032258064516), (9, 0.015873015873015872)]


In [7]:
pergunta = "o pedido nunca foi entregue"
resultado = buscador.buscar(pergunta, k=10, modo="hibrida")
print(f'"{pergunta}" — HÍBRIDA ({resultado.attrs["latencia_ms"]:.0f} ms)')
resultado[COLUNAS]

"o pedido nunca foi entregue" — HÍBRIDA (24 ms)


,pontuacao,similaridade,pos_vetorial,pos_bm25,nota,categoria,texto
rank,,,,,,,
1,0.0328,0.845,1,1,1,moveis_decoracao,o pedido nao foi entregue
2,0.0323,0.819,2,2,1,fashion_roupa_masculina,O pedido não foi entregue
3,0.0312,0.746,4,4,5,eletronicos,Não foi entregue o meu pedido
4,0.0303,0.724,7,5,1,bebidas,Produto não entregue. O pedido não foi entregue.
5,0.0303,0.725,6,6,2,cama_mesa_banho,Meu pedido ainda não foi. Meu pedido não foi entregue
6,0.0267,0.648,32,3,1,telefonia,Meu pedido não foi entregue De São Paulo SP foi para em Belém PA e nunca chegou aqui
7,0.0233,0.634,42,14,1,bebes,O pedido não foi entregue e consta no site como entregue.
8,0.0206,0.641,34,40,1,moveis_decoracao,Pedido não entregue
9,0.0159,0.759,3,<NA>,1,cama_mesa_banho,Meus pedidos não foram entregues.


**Conclusão da Seção 3:** no top-10 híbrido de "o pedido nunca foi entregue", as
primeiras posições são reviews que as **duas** buscas colocaram bem (`pos_vetorial` e
`pos_bm25` baixos). Paráfrases que só a vetorial achou ("Meus pedidos não foram
entregues.") ainda entram, mais abaixo.

## Seção 4 · Filtros por metadado

`posicoes_permitidas` transforma filtros de negócio em posições da base, e o `Buscador`
aplica o mesmo filtro **antes** de ranquear nas duas buscas (pré-filtro, como no
notebook 02). Assim dá para responder perguntas como:

- "O que dizem os clientes **cujo pedido atrasou**?"
- "Reclamações (**nota ≤ 2**) de **móveis** no **primeiro semestre de 2018**?"

In [8]:
mostrar_codigo(busca.posicoes_permitidas)

atrasados = busca.posicoes_permitidas(documentos, entregue_com_atraso=True)
print(f"\nPedidos entregues com atraso: {len(atrasados):,} reviews")
print(buscador.buscar("o que mais incomodou na compra", k=5, permitidos=atrasados)[COLUNAS].to_string())

moveis = busca.posicoes_permitidas(documentos, nota_max=2, categorias=["moveis_decoracao"],
                                   periodo=("2018-01", "2018-06"))
print(f"\nReclamações de móveis no 1º semestre de 2018: {len(moveis):,} reviews")
buscador.buscar("problema com a montagem ou peças", k=5, permitidos=moveis)[COLUNAS + ["ano_mes"]]

def posicoes_permitidas(documentos, nota_min=None, nota_max=None, categorias=None, estados=None,
                        periodo=None, entregue_com_atraso=None):
    """Posições dos documentos que passam nos filtros (None = sem filtro naquele campo).

    `periodo` é um par ("2018-01", "2018-06") de ano_mes, inclusivo.
    """
    filtro = pd.Series(True, index=documentos.index)
    if nota_min is not None:
        filtro &= documentos["nota"] >= nota_min
    if nota_max is not None:
        filtro &= documentos["nota"] <= nota_max
    if categorias is not None:
        filtro &= documentos["categoria"].isin(categorias)
    if estados is not None:
        filtro &= documentos["estado"].isin(estados)
    if periodo is not None:
        filtro &= documentos["ano_mes"].between(*periodo)
    if entregue_com_atraso is not None:
        filtro &= documentos["entregue_com_atraso"].fillna(not entregue_com_atraso) == entregue_com_atraso
    return np.flatnonzero(filtro.to_numpy())


Pedidos ent

,pontuacao,similaridade,pos_vetorial,pos_bm25,nota,categoria,texto,ano_mes
rank,,,,,,,,
1,0.0308,0.494,8,2,1,moveis_decoracao,"O produto veio incompleto, sem condições de montagem e uso",2018-03
2,0.0287,0.436,15,5,2,moveis_decoracao,"Tamanho. Faltou tamanho na descrição, acho q veio faltando peças, pois é muito baixa",2018-06
3,0.0283,0.547,2,22,1,moveis_decoracao,"Quadro quebrado. Infelizmente o meu quadro veio com a moldura quebrada, danificando a ...",2018-06
4,0.0282,0.570,1,25,2,moveis_decoracao,o produto chegou e como estava em viagem demorei uma semana para montar e quando fui m...,2018-03
5,0.0276,0.446,13,12,1,moveis_decoracao,"Luminárias. O produto veio faltando uma unidade, e ainda não consegui entrar em contat...",2018-06


**Conclusão da Seção 4:** com o filtro de atraso, a pergunta genérica "o que mais
incomodou na compra" já traz reclamações de demora; no recorte de móveis, as reviews
falam de peças faltando e produto quebrado, todas de 2018-03 a 2018-06.

## Seção 5 · Checagem de recuperação

Uma pergunta como "o pedido não chegou" tem **centenas** de reviews relevantes, então
não existe "o documento certo". Medimos a **precisão@10**: das 10 reviews devolvidas,
quantas falam do tema?

O "juiz" é uma expressão regular por pergunta (`avaliacao.PERGUNTAS_CHECAGEM`).
**Limite:** um juiz por palavras tende a **favorecer o BM25**. Para compensar, as
perguntas usam palavras diferentes das do padrão, e vale ler o top-k de olho. Esta é
uma checagem de sanidade da base de conhecimento; a avaliação completa, com gold set e
juiz LLM, está no notebook 10.

In [9]:
checagem = avaliacao.checar_recuperacao(buscador, k=10)
tabela = checagem.pivot(index="pergunta", columns="modo", values="precisao@10")[["vetorial", "bm25", "hibrida"]]
tabela.loc["MÉDIA"] = tabela.mean()
print("Precisão@10 (fração do top-10 que fala do tema):")
print(tabela.round(2).to_string())
print("\nLatência média (ms):", checagem.groupby("modo")["latencia_ms"].mean().round(0).to_dict())

Precisão@10 (fração do top-10 que fala do tema):
modo                                         vetorial  bm25  hibrida
pergunta                                                            
comprei mais de um item e só veio uma parte      0.50  0.70     0.40
embalagem chegou toda amassada                   0.90  1.00     0.90
entrega mais rápida do que o esperado            1.00  1.00     1.00
material de ótima qualidade, recomendo           1.00  1.00     1.00
o pedido nunca foi entregue                      0.80  0.70     0.80
problema com a nota fiscal da compra             0.90  1.00     1.00
produto chegou com defeito ou danificado         1.00  1.00     1.00
quero meu dinheiro de volta                      0.90  1.00     1.00
recebi um produto diferente do anunciado         0.90  1.00     1.00
vendedor não responde as mensagens               0.60  0.70     0.60
MÉDIA                                            0.85  0.91     0.87

Latência média (ms): {'bm25': 25.0, 'hibrida': 25.0, 

**Como ler:** a híbrida não precisa vencer em toda linha; o objetivo é ela não perder
feio em nenhuma, porque herda o melhor de cada busca. Repare nas linhas em que vetorial
e BM25 discordam: são as que justificam a fusão.

**Conclusão da Seção 5:** precisão@10 média de 0,85 (vetorial), 0,91 (BM25) e 0,87
(híbrida). A latência fica em dezenas de milissegundos nas três, desprezível perto de
uma chamada ao LLM. O BM25 leva vantagem aqui também porque o
juiz é por palavras. A pergunta mais difícil para todas é "comprei mais de um item e só
veio uma parte", em que as reviews descrevem a situação de muitas formas diferentes.

## Seção 6 · Sinal de recusa

O desafio exige **recusar** quando não há evidência. A busca sempre devolve um top-k,
então precisamos de um sinal: a **maior similaridade** entre a pergunta e as reviews
recuperadas. Perguntas do tema devem ficar bem acima de perguntas fora dele.

`config.LIMIAR_EVIDENCIA` (0,55) é o ponto de corte usado pelo pipeline RAG, junto com
a checagem de evidência pelo LLM (notebook 07), para decidir entre responder e
recusar.

In [10]:
recusa = avaliacao.checar_recusa(buscador)
print(recusa.to_string(index=False))
dentro = recusa.loc[recusa["grupo"] == "dentro", "similaridade_top1"]
fora = recusa.loc[recusa["grupo"] == "fora", "similaridade_top1"]
print(f"\nmenor similaridade DENTRO do tema: {dentro.min():.3f} | maior FORA do tema: {fora.max():.3f} "
      f"| limiar atual: {config.LIMIAR_EVIDENCIA}")

 grupo                                    pergunta  similaridade_top1  tem_evidencia
dentro                 o pedido nunca foi entregue              0.845           True
dentro    produto chegou com defeito ou danificado              0.921           True
dentro       entrega mais rápida do que o esperado              0.874           True
dentro    recebi um produto diferente do anunciado              0.997           True
dentro        problema com a nota fiscal da compra              0.818           True
dentro          vendedor não responde as mensagens              0.732           True
dentro comprei mais de um item e só veio uma parte              0.826           True
dentro                 quero meu dinheiro de volta              0.997           True
dentro              embalagem chegou toda amassada              0.795           True
dentro      material de ótima qualidade, recomendo              0.919           True
  fora                 qual é a capital da França?              0

**Conclusão da Seção 6:** a menor similaridade dentro do tema foi 0,73 e a maior fora
dele, 0,54. O limiar de 0,55 separa os dois grupos, mas com pouca folga do lado de
fora; por isso ele é só a primeira barreira, e o notebook 07 acrescenta um juiz LLM.

## Conclusões

| Decisão | Por quê |
|---|---|
| BM25 sobre `texto_lexical`, consulta com a mesma normalização | "não" e "nao" precisam casar; acentos variam entre clientes |
| BM25 reconstruído a cada carga, não gravado | 42 mil reviews curtas: frações de segundo, sem arquivo para sair de sincronia |
| RRF com k = 60 sobre os 50 primeiros de cada busca | usa posições, não pontuações em escalas diferentes; k = 60 é o valor do artigo original |
| Reviews com menos de 3 palavras não viram evidência | com pergunta curta, a vetorial as puxa com similaridade alta e sem sentido |
| Mesmo pré-filtro de metadado nas duas buscas | o filtro vale para a resposta inteira, não para uma das buscas |
| Precisão@10 com juiz por regex | sem gabarito único por pergunta; checagem de sanidade, não avaliação final |
| Similaridade do top-1 como sinal de recusa | a busca sempre devolve algo; o limiar é a primeira barreira |

**A base de conhecimento está pronta:** documentos com metadados (notebook 01), índice
vetorial (02) e busca híbrida com filtros (03).

**Próximo passo:** começa o **pipeline RAG**. O `04_guardrail_entrada.ipynb` cuida da
primeira etapa: validar a pergunta e bloquear PII e prompt injection antes de qualquer
chamada ao LLM.